## Naive Bayes — Gaussian Classification

---

### 1. Problem Setup

Assume we have a dataset  

$$
\{(\mathbf{x}_1, y_1), (\mathbf{x}_2, y_2), \dots, (\mathbf{x}_N, y_N)\}
$$  

where  

- $\mathbf{x}_i \in \mathbb{R}^D$ is the feature vector  
- $y_i \in \{1,2,\dots,K\}$ is the class label  

The goal is to classify each input into one of $K$ classes.

---

### 2. Probabilistic Model

Naive Bayes models the posterior probability using Bayes' theorem:

$$
P(y = k \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid y = k)\, P(y = k)}{P(\mathbf{x})}
$$

Since $P(\mathbf{x})$ is constant across classes, we use:

$$
P(y = k \mid \mathbf{x}) \propto P(\mathbf{x} \mid y = k)\, P(y = k)
$$

---

### 3. Naive Independence Assumption

Assume features are conditionally independent:

$$
P(\mathbf{x} \mid y = k) = \prod_{d=1}^{D} P(x_d \mid y = k)
$$

---

### 4. Gaussian Likelihood

Each feature is modeled as a Gaussian distribution:

$$
P(x_d \mid y = k) = \frac{1}{\sqrt{2\pi \sigma_{kd}^2}} 
\exp\left(-\frac{(x_d - \mu_{kd})^2}{2\sigma_{kd}^2}\right)
$$

where  

- $\mu_{kd}$ = mean of feature $d$ for class $k$  
- $\sigma_{kd}^2$ = variance of feature $d$ for class $k$  

---

### 5. Log-Likelihood Form

To avoid numerical underflow, we use log probabilities:

$$
\log P(y = k \mid \mathbf{x}) \propto \log P(y = k) + \sum_{d=1}^{D} \log P(x_d \mid y = k)
$$

Expanding:

$$
= \log \pi_k + \sum_{d=1}^{D} \left[
-\frac{1}{2} \log(2\pi \sigma_{kd}^2)
- \frac{(x_d - \mu_{kd})^2}{2\sigma_{kd}^2}
\right]
$$

---

### 6. Parameter Estimation

From training data:

Class prior:

$$
\pi_k = \frac{N_k}{N}
$$

Mean:

$$
\mu_{kd} = \frac{1}{N_k} \sum_{i: y_i = k} x_{i,d}
$$

Variance:

$$
\sigma_{kd}^2 = \frac{1}{N_k} \sum_{i: y_i = k} (x_{i,d} - \mu_{kd})^2 + \epsilon
$$

where $\epsilon$ is a small constant for numerical stability.

---

### 7. Prediction Rule

Predict the class with highest posterior:

$$
\hat{y} = \arg\max_k \left[
\log \pi_k + \sum_{d=1}^{D}
\left(
-\frac{1}{2}\log(2\pi\sigma_{kd}^2)
- \frac{(x_d - \mu_{kd})^2}{2\sigma_{kd}^2}
\right)
\right]
$$

---

### 8. Initialization

Parameters are initialized as:

$$
\pi_k, \quad \mu_{kd}, \quad \sigma_{kd}^2
$$

All computed directly from data (no iterative optimization required).

---

### 9. Training Procedure

1. Compute class priors:

$$
\pi_k = \frac{N_k}{N}
$$

2. Compute class-wise means:

$$
\mu_{kd}
$$

3. Compute class-wise variances:

$$
\sigma_{kd}^2
$$

---

### 10. Computational Notes

- Training is **very fast** (closed-form computation).  
- Prediction involves evaluating **log-probabilities**.  
- Adding $\epsilon$ prevents division by zero.  

---

### 11. Geometric Interpretation

- Each class is modeled as a **Gaussian distribution** in feature space.  
- Decision boundaries are **quadratic surfaces** when variances differ.  
- If variances are equal across classes:

$$
\sigma_{kd}^2 = \sigma^2
$$

the boundary becomes **linear**.

---

### 12. Prediction

For test data $\mathbf{x}_{\text{test}}$:

$$
\hat{y}_{\text{test}} = \arg\max_k \; \log P(y = k \mid \mathbf{x}_{\text{test}})
$$

---

### 13. Algorithm Summary

1. Compute class priors $\pi_k$  
2. Compute $\mu_{kd}$ and $\sigma_{kd}^2$  
3. For each test point:
   - Compute log-probability for each class  
   - Select class with maximum score  

---

### 14. Final Optimization Perspective

Naive Bayes performs:

$$
\hat{y} = \arg\max_k \; P(y = k)\prod_{d=1}^{D} P(x_d \mid y = k)
$$

which is equivalent to:

$$
\hat{y} = \arg\max_k \; \log P(y = k) + \sum_{d=1}^{D} \log P(x_d \mid y = k)
$$

This produces a **probabilistic classifier** assuming feature independence.

In [1]:
class NaiveBayes:
    """
    Gaussian Naive Bayes Classifier.

    This implementation assumes:
    - Features are conditionally independent given the class.
    - Each feature follows a Gaussian (normal) distribution.

    Parameters
    ----------
    eps : float, default=1e-10
        Small value added to variance for numerical stability.

    Attributes
    ----------
    pi_k : np.ndarray of shape (K,)
        Prior probabilities for each class.

    mu_kd : np.ndarray of shape (K, D)
        Mean of each feature for each class.

    sigma_sq_kd : np.ndarray of shape (K, D)
        Variance of each feature for each class.

    classes : np.ndarray of shape (K,)
        Unique class labels.

    num_classes : int
        Number of unique classes.
    """
    def __init__(self,eps=1e-10):
        self.eps = eps
        
        # Model parameters
        self.pi_k = None          # Class priors
        self.mu_kd = None         # Means
        self.sigma_sq_kd = None   # Variances

        self.num_classes = None
        self.classes = None

    def fit(self, X,y):

        """
        Fit the Naive Bayes model.

        Parameters
        ----------
        X : np.ndarray of shape (N, D)
            Feature matrix.

        y : np.ndarray of shape (N,)
            Target labels.

        Returns
        -------
        None
        """
        
        N,D = X.shape

        # Find unique classes and their counts
        self.classes,N_k = np.unique(y,return_counts=True)
        self.num_classes = len(self.classes)

        # Compute class priors
        self.pi_k = N_k/N

        # Compute mean for each class and feature
        self.mu_kd =  np.array([X[y==k].mean(axis=0) for k in self.classes]) 

        # Compute variance for each class and feature
        self.sigma_sq_kd  = np.array([((X[y==k] - self.mu_kd[i])**2).mean(axis=0)  for i,k in enumerate(self.classes)]) + self.eps 
        return None

    def predict(self,X):
        """
        Predict class labels for given input data.

        Parameters
        ----------
        X : np.ndarray of shape (N, D)
            Feature matrix.

        Returns
        -------
        y_pred : np.ndarray of shape (N,)
            Predicted class labels.
        """
        N, D = X.shape

        # Initialize score matrix (log probabilities)
        scores = np.zeros((N,self.num_classes))


        for k in range(self.num_classes):
            # Total score for class k
            scores[:, k] = np.log(self.pi_k[k]) + (-0.5 * np.log(2*np.pi*self.sigma_sq_kd[k]) - 
                                                  ((X-self.mu_kd[k])**2 / (2*self.sigma_sq_kd[k]))).sum(axis=1)

        # Select class with maximum score
        return self.classes[np.argmax(scores, axis=1)]
               



## 1. Purpose

The goal is to analyze how the **Naive Bayes assumption of feature independence** affects model performance.

Naive Bayes assumes:

$$
P(x_1, x_2, \dots, x_D \mid y) = \prod_{d=1}^{D} P(x_d \mid y)
$$

This implies that all features are **conditionally independent given the class label**.

---

## 2. Setup

Compare two datasets:

### Case 1: Independent Features

Features are sampled from a Gaussian distribution with **diagonal covariance**:

$$
x \sim \mathcal{N}(\mu_k, I)
$$



### Case 2: Correlated Features

Features are sampled from a Gaussian distribution with **non-diagonal covariance**:

$$
x \sim \mathcal{N}(\mu_k, \Sigma)
$$

where

$$
\Sigma =
\begin{bmatrix}
1 & \rho \\
\rho & 1
\end{bmatrix}, \quad \rho \neq 0
$$

This introduces **correlation between features**, violating the Naive Bayes assumption.

---

## 3. Model

Train a **Gaussian Naive Bayes classifier**, which models:

$$
P(x_d \mid y=k) \sim \mathcal{N}(\mu_{kd}, \sigma^2_{kd})
$$

Prediction is based on:

$$
\hat{y} = \arg\max_k \left[
\log P(y=k) + \sum_{d=1}^{D} \log P(x_d \mid y=k)
\right]
$$

---

## 4. Evaluation 

We evaluate performance using **classification accuracy**:

$$
\text{Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}(\hat{y}_i = y_i)
$$

---


## 5. Expected Outcome

- For **independent features**:
  
  $$
  \text{Naive Bayes performs well}
  $$

- For **correlated features**:

  $$
  \text{Accuracy decreases due to violated independence assumption}
  $$

---


In [2]:
import numpy as np


def generate_data(N=1500, correlated=False, rho=0.8):
    
    mu_pos = np.array([2, 2])
    mu_neg = np.array([-2, -2])

    if correlated:
        cov = np.array([[1, rho],
                        [rho, 1]])
    else:
        cov = np.eye(2)

    X_pos = np.random.multivariate_normal(mu_pos, cov, N)
    X_neg = np.random.multivariate_normal(mu_neg, cov, N)

    y_pos = np.ones(N)
    y_neg = -np.ones(N)

    X = np.vstack([X_pos, X_neg])
    y = np.hstack([y_pos, y_neg])

    return X, y

def train_test_split_manual(X, y, test_ratio=0.4):
    
    N = len(X)
    indices = np.arange(N)
    np.random.shuffle(indices)

    split = int(N * (1 - test_ratio))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def accuracy(y_true, y_pred):
    return np.round(np.mean(y_true == y_pred),4)


In [3]:
np.random.seed(42)


X_ind, y_ind = generate_data(correlated=False)

X_train, X_test, y_train, y_test = train_test_split_manual(X_ind, y_ind)

model_ind = NaiveBayes()
model_ind.fit(X_train, y_train)

y_pred_ind = model_ind.predict(X_test)
acc_ind = accuracy(y_test, y_pred_ind)



In [4]:
X_corr, y_corr = generate_data(correlated=True, rho=0.99)

X_train, X_test, y_train, y_test = train_test_split_manual(X_corr, y_corr)

model_corr = NaiveBayes()
model_corr.fit(X_train, y_train)

y_pred_corr = model_corr.predict(X_test)
acc_corr = accuracy(y_test, y_pred_corr)




In [5]:
print("Accuracy (Independent Features):", acc_ind)
print("Accuracy (Correlated Features):", acc_corr)

Accuracy (Independent Features): 0.9992
Accuracy (Correlated Features): 0.9758


---


## 6. Conclusion

Naive Bayes ignores feature dependencies:

$$
P(x_1, x_2 \mid y) \neq P(x_1 \mid y) P(x_2 \mid y)
$$

When correlation exists, the model **double counts information**, leading to suboptimal decision boundaries.

- Naive Bayes works best when features are **independent**  
- Performance degrades with **increasing correlation**  
- Despite this, Naive Bayes can still perform reasonably well due to its simplicity  

---